In [ ]:
# ============================================================
# rotina-principal.ipynb
# Orquestrador do pipeline completo de vinculacao
# ============================================================
from traceback import format_exc
try:
    from src.utils.gerenciador_sessao_spark_local import (
        GerenciadorSessaoSpark,
        ler_variavel_ambiente_local,
    )

    ambiente = ler_variavel_ambiente_local("AMBIENTE").upper()

    if ambiente != "MODELAGEM":
        ambiente = "PRODUCAO"

    gerenciador_spark = GerenciadorSessaoSpark(
        nome_sessao="mf_etl_vinculacao_meus_insights",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            # "SANDBOX": "t2i2016",
            "AMBIENTE": ambiente,
        },
        # nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        driver_memory="12g",
        driver_cores=4,
        executor_memory="12g",
        executor_cores=4,
        num_executors=8,
        jars=[
            "/dados/shared/bin/ojdbc8.jar",
        ],
        spark_conf={
            # memoria fora da JVM/container
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "4g",
            # serializer
            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",
            # adaptive query execution
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            # shuffle equilibrado
            "spark.sql.shuffle.partitions": "240",
            # escrita/leitura
            "spark.sql.sources.partitionOverwriteMode": "dynamic",
            "spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive": "true",
            # broadcast conservador
            "spark.sql.autoBroadcastJoinThreshold": "-1",
            "spark.sql.broadcastTimeout": "8000",
            # estabilidade de executores
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            # timezone
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    # get_ipython().display_formatter.formatters["text/plain"].for_type(__import__("ipywidgets").Widget, lambda *a, **k: None)
    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise

In [ ]:
try:
    print("type(spark)")
    print(type(spark))
except Exception as exc:
    print("ERRO:", type(exc).__name__, str(exc))

try:
    print("livy_session_id")
    print(spark.livy_session_id)
except Exception as exc:
    print("ERRO:", type(exc).__name__, str(exc))

try:
    print("bbmagic_session_id")
    print(spark.bbmagic_session_id)
except Exception as exc:
    print("ERRO:", type(exc).__name__, str(exc))

try:
    print("cluster")
    print(spark.cluster)
except Exception as exc:
    print("ERRO:", type(exc).__name__, str(exc))

try:
    print("spark_version")
    print(spark.spark_version)
except Exception as exc:
    print("ERRO:", type(exc).__name__, str(exc))

try:
    print("livy_url")
    print(spark.livy_url)
except Exception as exc:
    print("ERRO:", type(exc).__name__, str(exc))


In [ ]:
%%spark

try:
    print("APP_ID:")
    print(spark.sparkContext.applicationId)
except Exception as exc:
    print("ERRO APP_ID:", type(exc).__name__, str(exc))

try:
    print("\nMASTER:")
    print(spark.sparkContext.master)
except Exception as exc:
    print("ERRO MASTER:", type(exc).__name__, str(exc))

try:
    print("\nUI:")
    print(spark.sparkContext.uiWebUrl)
except Exception as exc:
    print("ERRO UI:", type(exc).__name__, str(exc))

try:
    print("\nVERSION:")
    print(spark.version)
except Exception as exc:
    print("ERRO VERSION:", type(exc).__name__, str(exc))

try:
    print("\nDEFAULT_PARALLELISM:")
    print(spark.sparkContext.defaultParallelism)
except Exception as exc:
    print("ERRO DEFAULT_PARALLELISM:", type(exc).__name__, str(exc))

In [ ]:
try:
    %run ./src/utils/gerenciador_sessao_spark_remoto.ipynb

    %run ./src/jobs/1_utils.ipynb
    %run ./src/jobs/2_contrato.ipynb
    %run ./src/jobs/3_preparacao_controle.ipynb
    %run ./src/jobs/4_sincronizacao.ipynb
    %run ./src/jobs/5_verificacao_origem.ipynb
    %run ./src/jobs/6_vinculacao.ipynb
    %run ./src/jobs/7_publicar_hive.ipynb
    %run ./src/jobs/8_publicar_oracle.ipynb
    %run ./src/jobs/9_encerrar_execucao.ipynb

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise

In [ ]:
%%spark

etapa_atual = "CONTEXTO"

try:
    ambiente = ler_variavel_ambiente_spark("AMBIENTE").upper()
    dominio = ler_variavel_ambiente_spark("DOMINIO").lower()
    hoje = validar_hoje_execucao(ler_variavel_ambiente_spark("HOJE"))

    primeira_carga = True
    executar = False
    detalhar_recomendacao = True

    if ambiente == "MODELAGEM":
        sandbox = ler_variavel_ambiente_spark("SANDBOX").lower()
        database = f"sbx_{sandbox}"
        path_save_hdfs = f"/dados/transientes/{dominio}/{sandbox}"
    else:
        database = f"hive_{dominio}"
        path_save_hdfs = f"/dados/corporativos/{dominio}"

    env_spark = dict(os.environ)

    cliente_oracle = criar_cliente_oracle_spark(env=env_spark)
    cliente_db2 = criar_cliente_db2_spark(env=env_spark)
    oracle_schema = cliente_oracle.schema

    logger_rotina.info(
        "[ROTINA][CONTEXTO] Contexto resolvido. "
        f"ambiente={ambiente} database={database} hoje={hoje} "
        f"path_save_hdfs={path_save_hdfs} primeira_carga={primeira_carga} executar={executar}"
    )

except Exception as exc:
    registrar_erro_rotina(etapa_atual, exc)
    raise

In [ ]:
%%spark

etapa_atual = "PREPARACAO_CONTROLE"

try:
    resultado_controle = executar_preparacao_controle(
        database=database,
        path_save_hdfs=path_save_hdfs,
        primeira_carga=primeira_carga,
        executar=executar,
        hoje=hoje,
    )

except Exception as exc:
    registrar_erro_rotina(etapa_atual, exc)
    raise

In [ ]:
%%spark

etapa_atual = "SINCRONIZACAO"

try:
    resultado_sincronizacao = executar_sincronizacao(
        df_ctl_oprl_rcm=resultado_controle["df_ctl_oprl_rcm"],
        database=database,
        oracle_schema=oracle_schema,
        cliente_oracle=cliente_oracle,
        cliente_db2=cliente_db2,
        hoje=hoje,
        primeira_carga=primeira_carga,
        detalhar_cenario=detalhar_recomendacao,
    )

except Exception as exc:
    registrar_erro_rotina(etapa_atual, exc)
    raise

In [ ]:
%%spark

etapa_atual = "VERIFICACAO_ORIGEM"

try:
    resultado_verificacao_origem = executar_verificacao_origem(
        df_ctl_oprl_rcm=resultado_sincronizacao["df_ctl_oprl_rcm"],
        cliente_db2=cliente_db2,
        detalhar_recomendacao=detalhar_recomendacao,
    )

except Exception as exc:
    registrar_erro_rotina(etapa_atual, exc)
    raise

In [ ]:
%%spark

etapa_atual = "VINCULACAO"

try:
    resultado_vinculacao = executar_vinculacao(
        df_ctl_oprl_rcm=resultado_verificacao_origem["df_ctl_oprl_rcm"],
        database=database,
        cliente_db2=cliente_db2,
        hoje=hoje,
        detalhar_recomendacao=detalhar_recomendacao,
    )

except Exception as exc:
    registrar_erro_rotina(etapa_atual, exc)
    raise

In [ ]:
%%spark

etapa_atual = "PUBLICACAO_HIVE"

try:
    resultado_publicacao_hive = publicar_hive(
        df_rcm_fnc_cli_final=resultado_vinculacao["df_rcm_fnc_cli_final"],
        df_rcm_fnc_cli_evtl_append=resultado_vinculacao["df_rcm_fnc_cli_evtl_append"],
        df_ctl_oprl_rcm_final=resultado_vinculacao["df_ctl_oprl_rcm"],
        database=database,
        path_save_hdfs=path_save_hdfs,
        executar=executar,
        detalhar_recomendacao=detalhar_recomendacao,
    )

except Exception as exc:
    registrar_erro_rotina(etapa_atual, exc)
    raise

In [ ]:
%%spark

etapa_atual = "ENCERRAMENTO_HIVE"

try:
    resultado_fluxo_hive = encerrar_fluxo_hive(
        path_save_hdfs=path_save_hdfs,
        executar=executar,
        publicacao_hive_concluida=resultado_publicacao_hive["hive_publicado"],
        stats={
            "controle": resultado_controle["stats"],
            "sincronizacao": resultado_sincronizacao["stats"],
            "verificacao_origem": resultado_verificacao_origem["stats"],
            "vinculacao": resultado_vinculacao["stats"],
            "publicacao_hive": resultado_publicacao_hive["stats"],
        },
        contexto={
            "executar": executar,
            "primeira_carga": primeira_carga,
            "hoje": hoje,
            "database": database,
            "path_save_hdfs": path_save_hdfs,
            "hive_publicado": resultado_publicacao_hive["hive_publicado"],
        },
    )

    limpeza_temporarios_ok = resultado_fluxo_hive["limpeza_temporarios_ok"]

except Exception as exc:
    registrar_erro_rotina(etapa_atual, exc)
    raise

In [ ]:
%%spark

etapa_atual = "DECISAO_PUBLICACAO_ORACLE"

try:
    resultado_publicacao_oracle = None
    status_oracle = None
    motivo_bloqueio_oracle = None

    if not executar:
        chamar_oracle = True
    elif not resultado_publicacao_hive["hive_publicado"]:
        chamar_oracle = False
        status_oracle = "BLOQUEADA_HIVE_NAO_PUBLICADO"
        motivo_bloqueio_oracle = "HIVE_NAO_PUBLICADO"
    elif not limpeza_temporarios_ok:
        chamar_oracle = False
        status_oracle = "BLOQUEADA_LIMPEZA_HDFS"
        motivo_bloqueio_oracle = "LIMPEZA_HDFS_NAO_CONCLUIDA"
    else:
        chamar_oracle = True

except Exception as exc:
    registrar_erro_rotina(etapa_atual, exc)
    raise

In [ ]:
%%spark

etapa_atual = "PUBLICACAO_ORACLE"

try:
    if chamar_oracle:
        resultado_publicacao_oracle = publicar_oracle(
            df_rcm_fnc_cli_final=resultado_vinculacao["df_rcm_fnc_cli_final"],
            database=database,
            oracle_schema=oracle_schema,
            cliente_oracle=cliente_oracle,
            executar=executar,
            hive_publicado=resultado_publicacao_hive["hive_publicado"],
            detalhar_recomendacao=detalhar_recomendacao,
        )

        stats_oracle = resultado_publicacao_oracle["stats"]
        if not executar:
            status_oracle = "DRY_RUN"
        elif stats_oracle.get("motivo") == "SEM_DIFERENCA_MATERIAL":
            status_oracle = "SEM_DIFERENCA_MATERIAL"
        elif stats_oracle.get("carga_executada") is True:
            status_oracle = "PUBLICADA"
        else:
            raise ValueError("Resultado inesperado da PUBLICACAO_ORACLE.")

except Exception as exc:
    status_oracle = "FALHA"
    resultado_final = dict(resultado_fluxo_hive)
    resultado_final["publicacao_oracle"] = None
    resultado_final["status_oracle"] = status_oracle
    resultado_final["motivo_bloqueio_oracle"] = None
    resultado_final["stats"]["publicacao_oracle"] = None
    logger_rotina.obj(resultado_final, title="[ROTINA][FIM_COM_FALHA_ORACLE] Resultado consolidado")
    registrar_erro_rotina(etapa_atual, exc)
    raise

In [ ]:
%%spark

etapa_atual = "RESULTADO_FINAL"

try:
    resultado_final = dict(resultado_fluxo_hive)
    resultado_final["publicacao_oracle"] = resultado_publicacao_oracle
    resultado_final["status_oracle"] = status_oracle
    resultado_final["motivo_bloqueio_oracle"] = motivo_bloqueio_oracle
    resultado_final["stats"]["publicacao_oracle"] = (
        resultado_publicacao_oracle["stats"]
        if resultado_publicacao_oracle is not None
        else None
    )

    logger_rotina.obj(resultado_final, title="[ROTINA][FIM] Resultado final")

except Exception as exc:
    registrar_erro_rotina(etapa_atual, exc)
    raise

In [ ]:
%%spark

spark.close()